# NOAA Station Launch Coverage Investigation

This notebook is rebuilt around the files in `data_final/NOAA_data`, which represent the NOAA/LCD files you were actually able to download.

The notebook answers four questions:

- Which launch facilities in the dataset now have a NOAA weather file in `data_final/NOAA_data`?
- Which station ids are actually present in those downloaded files?
- How much of each facility's launch history falls inside the downloaded weather date window?
- After converting launch times into local standard time, how many launches can be matched to a weather observation within 2 hours?

It also keeps a nearest-station reference table so you can compare the downloaded stations against the closest LCDv2 station inventory entries.

In [164]:
from pathlib import Path
import unicodedata

import numpy as np
import pandas as pd

DATA_DIR = Path("data_final")
NOAA_DIR = DATA_DIR / "NOAA_data"
OUTPUT_DIR = DATA_DIR / "derived"
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

STATION_LIST_PATH = NOAA_DIR / "lcdv2-station-list.txt"

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 200)

## Parse The LCDv2 Station Inventory

In [165]:
def load_lcdv2_station_list(path: Path) -> pd.DataFrame:
    rows = []
    with path.open("r", encoding="utf-8", errors="replace") as f:
        for raw in f:
            line = raw.rstrip("\n")
            if not line.strip():
                continue
            rows.append(
                {
                    "station_id": line[0:11].strip(),
                    "station_lat": pd.to_numeric(line[12:20].strip(), errors="coerce"),
                    "station_lon": pd.to_numeric(line[21:30].strip(), errors="coerce"),
                    "station_elevation_m": pd.to_numeric(line[31:37].strip(), errors="coerce"),
                    "station_name": line[41:].strip(),
                }
            )

    stations = pd.DataFrame(rows)
    stations = stations.dropna(subset=["station_lat", "station_lon"]).drop_duplicates("station_id").reset_index(drop=True)
    stations["station_country_hint"] = stations["station_id"].str[:2]
    return stations


stations = load_lcdv2_station_list(STATION_LIST_PATH)
print(f"Parsed {len(stations):,} stations from lcdv2-station-list.txt")
stations.head()

Parsed 24,072 stations from lcdv2-station-list.txt


,station_id,station_lat,station_lon,station_elevation_m,station_name,station_country_hint
0,ACL000BARA9,17.5910,-61.8210,5.0,BARBUDA,AC
1,ACM00078861,17.1167,-61.7833,10.0,COOLIDGE FIELD ANTIGUA (AUX.,AC
2,ACW00011647,17.1333,-61.7833,19.2,ST JOHNS,AC
3,AEI0000OMAA,24.4330,54.6511,26.8,ABU DHABI INTL,AE
4,AEI0000OMAB,23.6167,53.3833,94.0,BUHASA,AE


## Rebuild Launch Facility Summary From `data_final`

In [166]:
def infer_facility_group(row) -> str:
    location = str(row.get("Location", ""))
    country_code = str(row.get("Country_Code", "")).upper()
    combined = row.get("Comb Launch Site")

    if country_code != "US":
        return combined

    text = location.lower()
    if "kennedy space center" in text:
        return "Kennedy Space Center"
    if "cape canaveral" in text:
        return "Cape Canaveral Space Force Station"
    if "vandenberg" in text:
        return "Vandenberg Space Force Base"
    if "wallops" in text:
        return "Wallops Flight Facility"
    if "pacific spaceport" in text or "kodiak" in text:
        return "Pacific Spaceport Complex Alaska"
    if "china lake" in text:
        return "China Lake"
    if "edwards" in text:
        return "Edwards Air Force Base"
    if "mojave" in text:
        return "Mojave Air and Space Port"
    if "kauai" in text or "pacific missile range" in text:
        return "Pacific Missile Range Facility"
    return combined


launches = pd.read_csv(DATA_DIR / "Launches.csv")
locations = pd.read_csv(DATA_DIR / "Locations.csv")

launch_df = launches.copy()
launch_df["launch_time_utc"] = pd.to_datetime(launch_df["Launch Time"], utc=True, errors="coerce")
launch_df["launch_date"] = launch_df["launch_time_utc"].dt.date
launch_df = launch_df.merge(
    locations,
    left_on="Location",
    right_on="Orig_Addr",
    how="left",
    suffixes=("", "_location"),
)
launch_df["Country_Code"] = launch_df["Country_Code"].astype(str).str.upper()
launch_df["facility"] = launch_df.apply(infer_facility_group, axis=1)
launch_df["facility_lat_candidate"] = np.where(
    launch_df["Country_Code"].eq("US"),
    launch_df["Lat"],
    launch_df["Comb Launch Site Lat"],
)
launch_df["facility_lon_candidate"] = np.where(
    launch_df["Country_Code"].eq("US"),
    launch_df["Lon"],
    launch_df["Comb Launch Site Lon"],
)

facility_summary = (
    launch_df.dropna(subset=["facility", "facility_lat_candidate", "facility_lon_candidate"])
    .groupby(["facility", "Country", "Country_Code"], dropna=False)
    .agg(
        launches=("Launch Id", "count"),
        first_launch_utc=("launch_time_utc", "min"),
        last_launch_utc=("launch_time_utc", "max"),
        first_launch_date=("launch_date", "min"),
        last_launch_date=("launch_date", "max"),
        raw_location_strings=("Location", "nunique"),
        launch_site_labels=("Launch Site", "nunique"),
        facility_lat=("facility_lat_candidate", "mean"),
        facility_lon=("facility_lon_candidate", "mean"),
    )
    .reset_index()
    .rename(columns={"Country_Code": "country_code", "Country": "country"})
    .sort_values("launches", ascending=False)
    .reset_index(drop=True)
)

coordinate_overrides = {
    "Pacific Spaceport Complex Alaska": {
        "facility_lat": 57.4350,
        "facility_lon": -152.3390,
        "coordinate_note": "manual override: Kodiak-area PSCA coordinate; Locations.csv appears inland",
    }
}

facility_summary["coordinate_note"] = "Locations.csv"
for facility, override in coordinate_overrides.items():
    mask = facility_summary["facility"].eq(facility)
    facility_summary.loc[mask, "facility_lat"] = override["facility_lat"]
    facility_summary.loc[mask, "facility_lon"] = override["facility_lon"]
    facility_summary.loc[mask, "coordinate_note"] = override["coordinate_note"]

print(f"{len(facility_summary):,} facilities have coordinates")
print(f"{facility_summary['launches'].sum():,} launches map to those facilities")
facility_summary.head(20)

40 facilities have coordinates
6,168 launches map to those facilities


,facility,country,country_code,launches,first_launch_utc,last_launch_utc,first_launch_date,last_launch_date,raw_location_strings,launch_site_labels,facility_lat,facility_lon,coordinate_note
0,Plesetsk Cosmodrome,Russia,RU,1648,1966-03-17 10:28:00+00:00,2021-12-27 19:00:00+00:00,1966-03-17,2021-12-27,11,1,62.926415,40.555239,Locations.csv
1,Baikonur Cosmodrome,Kazakhstan,KZ,1525,1957-10-04 19:28:00+00:00,2021-12-27 13:10:00+00:00,1957-10-04,2021-12-27,21,1,45.964585,63.305243,Locations.csv
2,Cape Canaveral Space Force Station,United States,US,810,1957-12-06 16:44:00+00:00,2021-12-19 03:58:00+00:00,1957-12-06,2021-12-19,22,1,28.502515,-80.566468,Locations.csv
3,Vandenberg Space Force Base,United States,US,711,1959-02-28 21:49:00+00:00,2021-12-18 12:41:00+00:00,1959-02-28,2021-12-18,16,1,34.683069,-120.590025,Locations.csv
4,Guiana SC,French Guiana,GF,311,1970-03-10 12:20:00+00:00,2021-12-25 12:20:00+00:00,1970-03-10,2021-12-25,6,1,5.201590,-52.728131,Locations.csv
5,Kennedy Space Center,United States,US,192,1967-11-09 12:00:00+00:00,2021-12-21 10:07:00+00:00,1967-11-09,2021-12-21,2,1,28.626383,-80.620470,Locations.csv
6,Xichang Satellite LC,China,CN,167,1984-01-29 12:25:00+00:00,2021-12-29 16:43:00+00:00,1984-01-29,2021-12-29,3,1,27.893551,102.250931,Locations.csv
7,Jiuquan Satellite LC,China,CN,158,1970-04-24 13:35:00+00:00,2021-12-29 11:13:00+00:00,1970-04-24,2021-12-29,6,1,40.984524,100.208695,Locations.csv
8,Taiyuan Satellite LC,China,CN,99,1988-09-06 20:30:00+00:00,2021-12-26 03:11:00+00:00,1988-09-06,2021-12-26,4,1,38.848577,111.607983,Locations.csv
9,Kapustin Yar,Russia,RU,97,1961-10-27 16:30:00+00:00,1999-04-28 20:30:00+00:00,1961-10-27,1999-04-28,3,1,48.575267,45.766175,Locations.csv


## NOAA Files Actually Present In `data_final/NOAA_data`

In [167]:
FILE_TO_FACILITY = {
    "Alcantara_LC.csv": "Alcântara LC",
    "Baikonur_Cosmodrome.csv": "Baikonur Cosmodrome",
    "Base_Aerea_de_Gando.csv": "Base Aerea de Gando",
    "cape_canaveral_sfs.csv": "Cape Canaveral Space Force Station",
    "china_lake.csv": "China Lake",
    "edwards_afb.csv": "Edwards Air Force Base",
    "Guiana_SC.csv": "Guiana SC",
    "Imam_Khomeini_Spaceport.csv": "Imam Khomeini Spaceport",
    "Jiuquan_Satellite_LC.csv": "Jiuquan Satellite LC",
    "Kapustin_Yar.csv": "Kapustin Yar",
    "kennedy_sc.csv": "Kennedy Space Center",
    "Kiritimati_LA.csv": "Kiritimati LA",
    "Mahia_Peninsula.csv": "Māhia Peninsula",
    "Mojave_Air_and__Space_Port.csv": "Mojave Air and Space Port",
    "Naro_Space_Center.csv": "Naro Space Center",
    "pacific_missile_range_facility.csv": "Pacific Missile Range Facility",
    "Pacific_Spaceport_Complex_Alaska.csv": "Pacific Spaceport Complex Alaska",
    "Palmachim_Airbase.csv": "Palmachim Airbase",
    "Plesetsk_Cosmodrome.csv": "Plesetsk Cosmodrome",
    "RAAF_Woomera_RC.csv": "RAAF Woomera RC",
    "Ronald_Reagan_BMDTS.csv": "Ronald Reagan BMDTS",
    "Satish_Dhawan_SC.csv": "Satish Dhawan SC",
    "Shahrud_MTS.csv": "Shahrud MTS",
    "Sohae_SLS.csv": "Sohae SLS",
    "Svobodny_Cosmodrome.csv": "Svobodny Cosmodrome",
    "Taiyuan_Satellite_LC.csv": "Taiyuan Satellite LC",
    "Tanegashima_SC.csv": "Tanegashima SC",
    "Tonghae_SLG.csv": "Tonghae SLG",
    "Uchinoura_SC.csv": "Uchinoura SC",
    "vandenberg_sfb.csv": "Vandenberg Space Force Base",
    "Vostochny_Cosmodrome.csv": "Vostochny Cosmodrome",
    "wallops_flight_facility.csv": "Wallops Flight Facility",
    "Wenchang_Satellite_LC.csv": "Wenchang Satellite LC",
    "Xichang_Satellite_LC.csv": "Xichang Satellite LC",
    "Yasny_Cosmodrome.csv": "Yasny Cosmodrome",
}

FACILITY_UTC_OFFSET_HOURS = {
    "Alcântara LC": -3.0,
    "Baikonur Cosmodrome": 5.0,
    "Cape Canaveral Space Force Station": -5.0,
    "China Lake": -8.0,
    "Edwards Air Force Base": -8.0,
    "Guiana SC": -3.0,
    "Imam Khomeini Spaceport": 3.5,
    "Jiuquan Satellite LC": 8.0,
    "Kapustin Yar": 4.0,
    "Kennedy Space Center": -5.0,
    "Kiritimati LA": 14.0,
    "Māhia Peninsula": 12.0,
    "Mojave Air and Space Port": -8.0,
    "Naro Space Center": 9.0,
    "Pacific Missile Range Facility": -10.0,
    "Pacific Spaceport Complex Alaska": -9.0,
    "Palmachim Airbase": 2.0,
    "Plesetsk Cosmodrome": 3.0,
    "RAAF Woomera RC": 9.5,
    "Ronald Reagan BMDTS": 12.0,
    "Satish Dhawan SC": 5.5,
    "Shahrud MTS": 3.5,
    "Sohae SLS": 9.0,
    "Svobodny Cosmodrome": 9.0,
    "Taiyuan Satellite LC": 8.0,
    "Tanegashima SC": 9.0,
    "Tonghae SLG": 9.0,
    "Uchinoura SC": 9.0,
    "Vandenberg Space Force Base": -8.0,
    "Vostochny Cosmodrome": 9.0,
    "Wallops Flight Facility": -5.0,
    "Wenchang Satellite LC": 8.0,
    "Xichang Satellite LC": 8.0,
    "Yasny Cosmodrome": 5.0,
    "Base Aerea de Gando": 0.0,
}


def summarize_noaa_file(path: Path) -> dict:
    df = pd.read_csv(path, usecols=lambda c: c in {"STATION", "DATE"}, low_memory=False)
    dates = pd.to_datetime(df["DATE"], errors="coerce")
    station_ids = sorted(df["STATION"].dropna().astype(str).unique())
    return {
        "file_name": path.name,
        "facility": FILE_TO_FACILITY.get(path.name, path.stem),
        "file_path": str(path),
        "rows": len(df),
        "station_count": len(station_ids),
        "station_ids": ", ".join(station_ids),
        "weather_start_lstd": dates.min(),
        "weather_end_lstd": dates.max(),
    }


noaa_file_inventory = pd.DataFrame(
    [summarize_noaa_file(path) for path in sorted(NOAA_DIR.glob("*.csv"))]
).sort_values(["facility", "file_name"]).reset_index(drop=True)

noaa_file_inventory

,file_name,facility,file_path,rows,station_count,station_ids,weather_start_lstd,weather_end_lstd
0,Alcantara_LC.csv,Alcântara LC,data_final\NOAA_data\Alcantara_LC.csv,4296,1,BRM00082280,1997-01-01 09:00:00,2002-07-13 09:00:00
1,Baikonur_Cosmodrome.csv,Baikonur Cosmodrome,data_final\NOAA_data\Baikonur_Cosmodrome.csv,13176,1,KZM00035953,1966-01-01 02:00:00,1991-12-21 02:00:00
2,Base_Aerea_de_Gando.csv,Base Aerea de Gando,data_final\NOAA_data\Base_Aerea_de_Gando.csv,4,1,SPM00060036,1997-01-01 00:00:00,1997-01-01 18:00:00
3,cape_canaveral_sfs.csv,Cape Canaveral Space Force Station,data_final\NOAA_data\cape_canaveral_sfs.csv,517229,2,"USL000TRDF1, USW00012868",1957-12-01 00:00:00,2021-12-31 23:56:00
4,china_lake.csv,China Lake,data_final\NOAA_data\china_lake.csv,9135,1,USW00093104,1958-01-01 00:00:00,1958-12-31 23:59:00
5,edwards_afb.csv,Edwards Air Force Base,data_final\NOAA_data\edwards_afb.csv,41606,1,USW00023114,1990-01-02 05:10:00,1994-12-30 16:00:00
6,Guiana_SC.csv,Guiana SC,data_final\NOAA_data\Guiana_SC.csv,483259,1,FGI0000SOCA,1972-12-31 23:00:00,2021-12-31 23:30:00
7,Imam_Khomeini_Spaceport.csv,Imam Khomeini Spaceport,data_final\NOAA_data\Imam_Khomeini_Spaceport.csv,90207,1,IRI0000OIIS,2008-01-01 00:30:00,2021-12-31 21:30:00
8,Jiuquan_Satellite_LC.csv,Jiuquan Satellite LC,data_final\NOAA_data\Jiuquan_Satellite_LC.csv,54727,1,CHA00524460,1973-01-01 08:00:00,2002-05-13 17:00:00
9,Kapustin_Yar.csv,Kapustin Yar,data_final\NOAA_data\Kapustin_Yar.csv,23264,1,RSM00034571,1961-01-01 03:00:00,1974-03-10 00:00:00


## Downloaded Stations In Those Files

In [168]:
def haversine_miles(lat1, lon1, lat2, lon2):
    radius_miles = 3958.7613
    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    return 2 * radius_miles * np.arcsin(np.sqrt(a))


downloaded_station_rows = []
for _, row in noaa_file_inventory.iterrows():
    facility = row["facility"]
    facility_row = facility_summary.loc[facility_summary["facility"] == facility]
    if facility_row.empty:
        continue
    facility_row = facility_row.iloc[0]

    station_ids = [s.strip() for s in str(row["station_ids"]).split(",") if s.strip()]
    for station_id in station_ids:
        station_row = stations.loc[stations["station_id"] == station_id]
        if station_row.empty:
            downloaded_station_rows.append(
                {
                    "facility": facility,
                    "file_name": row["file_name"],
                    "station_id": station_id,
                    "station_found_in_inventory": False,
                }
            )
            continue

        station_row = station_row.iloc[0]
        distance = haversine_miles(
            facility_row["facility_lat"],
            facility_row["facility_lon"],
            station_row["station_lat"],
            station_row["station_lon"],
        )
        all_distances = haversine_miles(
            facility_row["facility_lat"],
            facility_row["facility_lon"],
            stations["station_lat"].to_numpy(),
            stations["station_lon"].to_numpy(),
        )
        rank = int((all_distances < distance).sum() + 1)
        downloaded_station_rows.append(
            {
                "facility": facility,
                "country_code": facility_row["country_code"],
                "launches": facility_row["launches"],
                "file_name": row["file_name"],
                "station_id": station_id,
                "station_name": station_row["station_name"],
                "station_lat": station_row["station_lat"],
                "station_lon": station_row["station_lon"],
                "station_elevation_m": station_row["station_elevation_m"],
                "distance_miles": distance,
                "distance_rank_among_all_lcdv2_stations": rank,
                "station_found_in_inventory": True,
            }
        )

downloaded_station_summary = pd.DataFrame(downloaded_station_rows).sort_values(
    ["launches", "facility", "distance_miles"], ascending=[False, True, True]
).reset_index(drop=True)

downloaded_station_summary

,facility,country_code,launches,file_name,station_id,station_name,station_lat,station_lon,station_elevation_m,distance_miles,distance_rank_among_all_lcdv2_stations,station_found_in_inventory
0,Plesetsk Cosmodrome,RU,1648,Plesetsk_Cosmodrome.csv,RSM00022657,EMCA,63.0670,40.3500,108.0,11.653629,1,True
1,Baikonur Cosmodrome,KZ,1525,Baikonur_Cosmodrome.csv,KZM00035953,DZUSALY,45.5000,64.1000,101.0,49.995056,1,True
2,Cape Canaveral Space Force Station,US,810,cape_canaveral_sfs.csv,USW00012868,CAPE KENNEDY AFS 74794,28.4833,-80.5667,3.0,1.327739,1,True
3,Cape Canaveral Space Force Station,US,810,cape_canaveral_sfs.csv,USL000TRDF1,TRIDENT PIER,28.4200,-80.5800,10.0,5.760230,2,True
4,Vandenberg Space Force Base,US,711,vandenberg_sfb.csv,USW00093215,POINT ARGUELLO AFWB,34.6667,-120.5833,111.9,1.193786,1,True
5,Vandenberg Space Force Base,US,711,vandenberg_sfb.csv,USW00093214,VANDENBERG AFB 72393,34.7167,-120.5667,112.5,2.674893,2,True
6,Guiana SC,GF,311,Guiana_SC.csv,FGI0000SOCA,ROCHAMBEAU,4.8198,-52.3604,7.9,36.558001,1,True
7,Kennedy Space Center,US,192,kennedy_sc.csv,USW00012886,TITUSVILLE NASA SHUTTLE FAC,28.6167,-80.6833,3.0,3.868927,1,True
8,Kennedy Space Center,US,192,kennedy_sc.csv,USW00092821,TITUSVILLE 7 E,28.6158,-80.6928,0.9,4.447363,2,True
9,Xichang Satellite LC,CN,167,Xichang_Satellite_LC.csv,CHM00056571,XICHANG,27.9000,102.2667,1599.0,1.060992,1,True


## Distance Between Launch Facilities And Downloaded Weather Stations

In [169]:
downloaded_station_distance_summary = (
    downloaded_station_summary.groupby(["facility", "country_code", "launches"], dropna=False)
    .agg(
        downloaded_station_count=("station_id", "nunique"),
        nearest_downloaded_station_id=("station_id", "first"),
        nearest_downloaded_station_name=("station_name", "first"),
        nearest_downloaded_station_distance_miles=("distance_miles", "min"),
        farthest_downloaded_station_distance_miles=("distance_miles", "max"),
        best_distance_rank_among_all_lcdv2_stations=("distance_rank_among_all_lcdv2_stations", "min"),
    )
    .reset_index()
    .sort_values(["launches", "nearest_downloaded_station_distance_miles"], ascending=[False, True])
    .reset_index(drop=True)
)

downloaded_station_distance_summary

,facility,country_code,launches,downloaded_station_count,nearest_downloaded_station_id,nearest_downloaded_station_name,nearest_downloaded_station_distance_miles,farthest_downloaded_station_distance_miles,best_distance_rank_among_all_lcdv2_stations
0,Plesetsk Cosmodrome,RU,1648,1,RSM00022657,EMCA,11.653629,11.653629,1
1,Baikonur Cosmodrome,KZ,1525,1,KZM00035953,DZUSALY,49.995056,49.995056,1
2,Cape Canaveral Space Force Station,US,810,2,USW00012868,CAPE KENNEDY AFS 74794,1.327739,5.760230,1
3,Vandenberg Space Force Base,US,711,2,USW00093215,POINT ARGUELLO AFWB,1.193786,2.674893,1
4,Guiana SC,GF,311,1,FGI0000SOCA,ROCHAMBEAU,36.558001,36.558001,1
5,Kennedy Space Center,US,192,2,USW00012886,TITUSVILLE NASA SHUTTLE FAC,3.868927,4.447363,1
6,Xichang Satellite LC,CN,167,1,CHM00056571,XICHANG,1.060992,1.060992,1
7,Jiuquan Satellite LC,CN,158,1,CHA00524460,SHUANGCHENGTZU,45.711034,45.711034,1
8,Taiyuan Satellite LC,CN,99,1,CHM00053663,WU-ZHAI,12.993011,12.993011,2
9,Kapustin Yar,RU,97,1,RSM00034571,KAPUSTIN JAR,1.861946,1.861946,1


## Three Closest LCDv2 Stations For Every Launch Facility

In [170]:
downloaded_station_pairs = {
    (row["facility"], row["station_id"])
    for _, row in downloaded_station_summary.dropna(subset=["station_id"]).iterrows()
}

nearest_rows = []
for _, row in facility_summary.iterrows():
    distances = haversine_miles(
        row["facility_lat"],
        row["facility_lon"],
        stations["station_lat"].to_numpy(),
        stations["station_lon"].to_numpy(),
    )
    nearest = stations.copy()
    nearest["distance_miles"] = distances
    nearest = nearest.nsmallest(3, "distance_miles").copy()
    nearest["facility"] = row["facility"]
    nearest["country_code"] = row["country_code"]
    nearest["launches"] = row["launches"]
    nearest["first_launch_date"] = row["first_launch_date"]
    nearest["last_launch_date"] = row["last_launch_date"]
    nearest["facility_lat"] = row["facility_lat"]
    nearest["facility_lon"] = row["facility_lon"]
    nearest["coordinate_note"] = row["coordinate_note"]
    nearest["distance_rank"] = np.arange(1, len(nearest) + 1)
    nearest_rows.append(nearest)

three_closest_stations = pd.concat(nearest_rows, ignore_index=True)
three_closest_stations["downloaded_in_data_final"] = [
    (facility, station_id) in downloaded_station_pairs
    for facility, station_id in zip(
        three_closest_stations["facility"],
        three_closest_stations["station_id"],
    )
]

three_closest_stations = three_closest_stations[
    [
        "facility",
        "country_code",
        "launches",
        "first_launch_date",
        "last_launch_date",
        "facility_lat",
        "facility_lon",
        "coordinate_note",
        "distance_rank",
        "station_id",
        "station_name",
        "station_country_hint",
        "station_lat",
        "station_lon",
        "station_elevation_m",
        "distance_miles",
        "downloaded_in_data_final",
    ]
].sort_values(["launches", "facility", "distance_rank"], ascending=[False, True, True]).reset_index(drop=True)

three_closest_stations

,facility,country_code,launches,first_launch_date,last_launch_date,facility_lat,facility_lon,coordinate_note,distance_rank,station_id,station_name,station_country_hint,station_lat,station_lon,station_elevation_m,distance_miles,downloaded_in_data_final
0,Plesetsk Cosmodrome,RU,1648,1966-03-17,2021-12-27,62.926415,40.555239,Locations.csv,1,RSM00022657,EMCA,RS,63.0670,40.3500,108.0,11.653629,True
1,Plesetsk Cosmodrome,RU,1648,1966-03-17,2021-12-27,62.926415,40.555239,Locations.csv,2,RSMU0022648,TURCHASOVO USSR,RS,63.1200,39.2800,35.0,42.147338,False
2,Plesetsk Cosmodrome,RU,1648,1966-03-17,2021-12-27,62.926415,40.555239,Locations.csv,3,RSMU0022656,YEMETSK USSR (EMETSK),RS,63.4700,41.7500,28.0,52.876649,False
3,Baikonur Cosmodrome,KZ,1525,1957-10-04,2021-12-27,45.964585,63.305243,Locations.csv,1,KZM00035953,DZUSALY,KZ,45.5000,64.1000,101.0,49.995056,True
4,Baikonur Cosmodrome,KZ,1525,1957-10-04,2021-12-27,45.964585,63.305243,Locations.csv,2,KZM00035849,KAZALY,KZ,45.7667,62.1167,68.0,58.795241,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,Shahrud MTS,IR,1,2020-04-22,2020-04-22,36.406224,55.016269,Locations.csv,2,IRI0000OING,GORGAN,IR,36.9094,54.4013,-7.3,48.688217,False
116,Shahrud MTS,IR,1,2020-04-22,2020-04-22,36.406224,55.016269,Locations.csv,3,IRI0000OINE,KALALEH,IR,37.3833,55.4500,152.4,71.637226,False
117,Tai Rui Barge,NaN,1,2019-06-05,2019-06-05,35.494277,123.796461,Locations.csv,1,KSM00047169,HEUKSANDO,KS,34.6872,125.4510,68.5,108.897648,False
118,Tai Rui Barge,NaN,1,2019-06-05,2019-06-05,35.494277,123.796461,Locations.csv,2,KSA00471963,YEONPYEONGDO,KS,34.6670,125.6830,-999.0,121.015068,False


## Launch Date-Window Coverage From The Downloaded NOAA Files

In [171]:
facility_coverage_rows = []
noaa_inventory_by_facility = noaa_file_inventory.set_index("facility")

for _, facility_row in facility_summary.iterrows():
    facility = facility_row["facility"]
    launches_for_facility = launch_df.loc[launch_df["facility"] == facility].copy()
    launches_for_facility = launches_for_facility.dropna(subset=["launch_time_utc"]).copy()

    offset = FACILITY_UTC_OFFSET_HOURS.get(facility)
    launches_for_facility["launch_time_lstd"] = (
        launches_for_facility["launch_time_utc"] + pd.to_timedelta(offset, unit="h")
    ).dt.tz_localize(None) if offset is not None else pd.NaT

    if facility in noaa_inventory_by_facility.index:
        inv = noaa_inventory_by_facility.loc[facility]
        weather_start = inv["weather_start_lstd"]
        weather_end = inv["weather_end_lstd"]
        in_window = launches_for_facility["launch_time_lstd"].between(weather_start, weather_end, inclusive="both")
        before_window = launches_for_facility["launch_time_lstd"] < weather_start
        after_window = launches_for_facility["launch_time_lstd"] > weather_end
        file_exists = True
        station_ids = inv["station_ids"]
        file_name = inv["file_name"]
    else:
        weather_start = pd.NaT
        weather_end = pd.NaT
        in_window = pd.Series(False, index=launches_for_facility.index)
        before_window = pd.Series(False, index=launches_for_facility.index)
        after_window = pd.Series(False, index=launches_for_facility.index)
        file_exists = False
        station_ids = ""
        file_name = ""

    facility_coverage_rows.append(
        {
            "facility": facility,
            "country_code": facility_row["country_code"],
            "launches": facility_row["launches"],
            "first_launch_date": facility_row["first_launch_date"],
            "last_launch_date": facility_row["last_launch_date"],
            "file_exists": file_exists,
            "file_name": file_name,
            "station_ids": station_ids,
            "utc_offset_hours": offset,
            "weather_start_lstd": weather_start,
            "weather_end_lstd": weather_end,
            "launches_in_weather_window": int(in_window.sum()),
            "launches_before_weather_start": int(before_window.sum()),
            "launches_after_weather_end": int(after_window.sum()),
            "date_window_coverage_rate": float(in_window.mean()) if len(launches_for_facility) else np.nan,
        }
    )

facility_date_coverage = pd.DataFrame(facility_coverage_rows).sort_values(
    ["launches", "file_exists", "launches_in_weather_window"], ascending=[False, False, False]
).reset_index(drop=True)

facility_date_coverage

,facility,country_code,launches,first_launch_date,last_launch_date,file_exists,file_name,station_ids,utc_offset_hours,weather_start_lstd,weather_end_lstd,launches_in_weather_window,launches_before_weather_start,launches_after_weather_end,date_window_coverage_rate
0,Plesetsk Cosmodrome,RU,1648,1966-03-17,2021-12-27,True,Plesetsk_Cosmodrome.csv,RSM00022657,3.0,1966-01-01 00:00:00,1988-10-08 15:00:00,1240,0,408,0.752427
1,Baikonur Cosmodrome,KZ,1525,1957-10-04,2021-12-27,True,Baikonur_Cosmodrome.csv,KZM00035953,5.0,1966-01-01 02:00:00,1991-12-21 02:00:00,843,128,554,0.552787
2,Cape Canaveral Space Force Station,US,810,1957-12-06,2021-12-19,True,cape_canaveral_sfs.csv,"USL000TRDF1, USW00012868",-5.0,1957-12-01 00:00:00,2021-12-31 23:56:00,810,0,0,1.000000
3,Vandenberg Space Force Base,US,711,1959-02-28,2021-12-18,True,vandenberg_sfb.csv,"USW00093214, USW00093215",-8.0,1959-01-01 00:00:00,2021-12-31 23:58:00,711,0,0,1.000000
4,Guiana SC,GF,311,1970-03-10,2021-12-25,True,Guiana_SC.csv,FGI0000SOCA,-3.0,1972-12-31 23:00:00,2021-12-31 23:30:00,306,5,0,0.983923
5,Kennedy Space Center,US,192,1967-11-09,2021-12-21,True,kennedy_sc.csv,"USW00012886, USW00092821",-5.0,1978-03-16 19:00:00,2021-12-31 23:59:00,175,17,0,0.911458
6,Xichang Satellite LC,CN,167,1984-01-29,2021-12-29,True,Xichang_Satellite_LC.csv,CHM00056571,8.0,1984-01-01 00:00:00,2021-12-31 23:59:00,167,0,0,1.000000
7,Jiuquan Satellite LC,CN,158,1970-04-24,2021-12-29,True,Jiuquan_Satellite_LC.csv,CHA00524460,8.0,1973-01-01 08:00:00,2002-05-13 17:00:00,29,2,127,0.183544
8,Taiyuan Satellite LC,CN,99,1988-09-06,2021-12-26,True,Taiyuan_Satellite_LC.csv,CHM00053663,8.0,2001-11-18 08:00:00,2007-07-27 20:00:00,11,12,76,0.111111
9,Kapustin Yar,RU,97,1961-10-27,1999-04-28,True,Kapustin_Yar.csv,RSM00034571,4.0,1961-01-01 03:00:00,1974-03-10 00:00:00,75,0,22,0.773196


## Nearest-Observation Weather Match Coverage Within 2 Hours

In [172]:
WEATHER_NUMERIC_COLUMNS = [
    "HourlyAltimeterSetting",
    "HourlyDryBulbTemperature",
    "HourlyDewPointTemperature",
    "HourlyRelativeHumidity",
    "HourlyPrecipitation",
    "HourlyVisibility",
    "HourlyStationPressure",
    "HourlySeaLevelPressure",
    "HourlyWindSpeed",
    "HourlyWindDirection",
    "HourlyWindGustSpeed",
    "HourlyWetBulbTemperature",
]

WEATHER_TEXT_COLUMNS = [
    "HourlyPresentWeatherType",
    "HourlySkyConditions",
]

SHORT_DURATION_PRECIP_COLUMNS = [
    "HourlyPrecipitation",
    "HourlyPrecipitation01Hour",
    "HourlyPrecipitation03Hour",
    "HourlyPrecipitation06Hour",
]


def clean_lcd_numeric(series: pd.Series) -> pd.Series:
    return pd.to_numeric(
        series.astype(str).str.extract(r"([-+]?[0-9]*\.?[0-9]+)")[0],
        errors="coerce",
    )


def load_best_hourly_weather(path: Path) -> pd.DataFrame:
    weather_raw = pd.read_csv(path, low_memory=False)
    keep_cols = ["STATION", "DATE", "REPORT_TYPE"] + [
        c for c in WEATHER_NUMERIC_COLUMNS + WEATHER_TEXT_COLUMNS + SHORT_DURATION_PRECIP_COLUMNS
        if c in weather_raw.columns
    ]
    keep_cols = list(dict.fromkeys(keep_cols))
    weather = weather_raw[keep_cols].copy()
    weather = weather.loc[:, ~weather.columns.duplicated()].copy()

    for col in [c for c in WEATHER_NUMERIC_COLUMNS + SHORT_DURATION_PRECIP_COLUMNS if c in weather.columns]:
        weather[col] = clean_lcd_numeric(weather[col])

    weather["weather_obs_time_lstd"] = pd.to_datetime(weather["DATE"], errors="coerce")
    weather = weather.dropna(subset=["weather_obs_time_lstd"]).copy()
    weather["weather_obs_time_lstd"] = weather["weather_obs_time_lstd"].astype("datetime64[ns]")

    numeric_cols = [c for c in WEATHER_NUMERIC_COLUMNS if c in weather.columns]
    text_cols = [c for c in WEATHER_TEXT_COLUMNS if c in weather.columns]

    weather["hourly_nonnulls"] = weather[numeric_cols].notna().sum(axis=1)
    for col in text_cols:
        weather["hourly_nonnulls"] += weather[col].notna().astype(int)

    weather = (
        weather.loc[weather["hourly_nonnulls"] > 0]
        .sort_values(["weather_obs_time_lstd", "hourly_nonnulls"], ascending=[True, False])
        .drop_duplicates(subset=["weather_obs_time_lstd"], keep="first")
        .sort_values("weather_obs_time_lstd")
        .reset_index(drop=True)
    )
    return weather


weather_match_rows = []
matched_launch_frames = []

for _, inv in noaa_file_inventory.sort_values("facility").iterrows():
    facility = inv["facility"]
    offset = FACILITY_UTC_OFFSET_HOURS.get(facility)
    if offset is None:
        continue

    facility_launches = launch_df.loc[launch_df["facility"] == facility].copy()
    facility_launches = facility_launches.dropna(subset=["launch_time_utc"]).copy()
    if facility_launches.empty:
        continue

    facility_launches["launch_time_lstd"] = (
        facility_launches["launch_time_utc"] + pd.to_timedelta(offset, unit="h")
    ).dt.tz_localize(None)
    facility_launches["launch_time_lstd"] = facility_launches["launch_time_lstd"].astype("datetime64[ns]")
    facility_launches = facility_launches.sort_values("launch_time_lstd")

    weather = load_best_hourly_weather(Path(inv["file_path"]))
    if weather.empty:
        weather_match_rows.append(
            {
                "facility": facility,
                "launches": len(facility_launches),
                "matched_launches": 0,
                "match_rate": 0.0,
                "median_abs_diff_minutes": np.nan,
                "weather_rows_after_cleaning": 0,
                "weather_start_lstd": pd.NaT,
                "weather_end_lstd": pd.NaT,
                "station_ids": inv["station_ids"],
                "file_name": inv["file_name"],
            }
        )
        continue

    merged = pd.merge_asof(
        facility_launches,
        weather,
        left_on="launch_time_lstd",
        right_on="weather_obs_time_lstd",
        direction="nearest",
        tolerance=pd.Timedelta("2h"),
    )

    merged["facility"] = facility
    merged["weather_matched"] = merged["weather_obs_time_lstd"].notna()
    merged["weather_time_diff_minutes"] = (
        (merged["launch_time_lstd"] - merged["weather_obs_time_lstd"]).dt.total_seconds().abs() / 60
    )
    merged["file_name"] = inv["file_name"]
    merged["station_ids"] = inv["station_ids"]
    matched_launch_frames.append(merged)

    weather_match_rows.append(
        {
            "facility": facility,
            "launches": len(facility_launches),
            "matched_launches": int(merged["weather_matched"].sum()),
            "match_rate": float(merged["weather_matched"].mean()),
            "median_abs_diff_minutes": merged.loc[merged["weather_matched"], "weather_time_diff_minutes"].median(),
            "weather_rows_after_cleaning": len(weather),
            "weather_start_lstd": weather["weather_obs_time_lstd"].min(),
            "weather_end_lstd": weather["weather_obs_time_lstd"].max(),
            "station_ids": inv["station_ids"],
            "file_name": inv["file_name"],
        }
    )

weather_match_coverage = pd.DataFrame(weather_match_rows).sort_values(
    ["matched_launches", "match_rate"], ascending=[False, False]
).reset_index(drop=True)

matched_launch_weather = pd.concat(matched_launch_frames, ignore_index=True) if matched_launch_frames else pd.DataFrame()

weather_match_coverage

,facility,launches,matched_launches,match_rate,median_abs_diff_minutes,weather_rows_after_cleaning,weather_start_lstd,weather_end_lstd,station_ids,file_name
0,Plesetsk Cosmodrome,1648,1091,0.662015,45.0,57732,1966-01-01 00:00:00,1988-10-08 15:00:00,RSM00022657,Plesetsk_Cosmodrome.csv
1,Vandenberg Space Force Base,711,672,0.945148,13.0,624228,1959-01-01 00:00:00,2021-12-31 23:58:00,"USW00093214, USW00093215",vandenberg_sfb.csv
2,Cape Canaveral Space Force Station,810,509,0.628395,14.0,459992,1957-12-01 00:00:00,2021-12-31 23:56:00,"USL000TRDF1, USW00012868",cape_canaveral_sfs.csv
3,Guiana SC,311,304,0.977492,11.0,483251,1972-12-31 23:00:00,2021-12-31 23:30:00,FGI0000SOCA,Guiana_SC.csv
4,Kennedy Space Center,192,173,0.901042,0.0,1978955,1978-03-16 19:00:00,2021-12-31 23:56:00,"USW00012886, USW00092821",kennedy_sc.csv
5,Xichang Satellite LC,167,167,1.000000,53.0,110869,1984-01-01 02:00:00,2021-12-31 23:00:00,CHM00056571,Xichang_Satellite_LC.csv
6,Baikonur Cosmodrome,1525,145,0.095082,50.0,13176,1966-01-01 02:00:00,1991-12-21 02:00:00,KZM00035953,Baikonur_Cosmodrome.csv
7,Tanegashima SC,85,67,0.788235,21.0,201486,1975-01-01 08:00:00,2021-12-31 18:00:00,JAI0000RJFG,Tanegashima_SC.csv
8,Kapustin Yar,97,54,0.556701,58.0,23264,1961-01-01 03:00:00,1974-03-10 00:00:00,RSM00034571,Kapustin_Yar.csv
9,Wallops Flight Facility,49,35,0.714286,13.0,397796,1966-10-01 07:00:00,2021-12-31 23:54:00,USW00093739,wallops_flight_facility.csv


## Join Quality Focus

In [173]:
join_quality_summary = (
    facility_date_coverage.merge(
        weather_match_coverage[
            [
                "facility",
                "matched_launches",
                "match_rate",
                "median_abs_diff_minutes",
                "weather_rows_after_cleaning",
            ]
        ],
        on="facility",
        how="left",
    )
    .merge(
        downloaded_station_distance_summary[
            [
                "facility",
                "downloaded_station_count",
                "nearest_downloaded_station_id",
                "nearest_downloaded_station_name",
                "nearest_downloaded_station_distance_miles",
                "best_distance_rank_among_all_lcdv2_stations",
            ]
        ],
        on="facility",
        how="left",
    )
)

join_quality_summary["window_to_match_conversion_rate"] = (
    join_quality_summary["matched_launches"] / join_quality_summary["launches_in_weather_window"]
)
join_quality_summary["unmatched_launches_inside_window"] = (
    join_quality_summary["launches_in_weather_window"] - join_quality_summary["matched_launches"]
)
join_quality_summary["outside_weather_window_launches"] = (
    join_quality_summary["launches"] - join_quality_summary["launches_in_weather_window"]
)

join_quality_summary = join_quality_summary.sort_values(
    ["launches", "match_rate", "date_window_coverage_rate"],
    ascending=[False, False, False],
).reset_index(drop=True)

join_quality_summary[
    [
        "facility",
        "country_code",
        "launches",
        "station_ids",
        "nearest_downloaded_station_id",
        "nearest_downloaded_station_distance_miles",
        "launches_in_weather_window",
        "date_window_coverage_rate",
        "matched_launches",
        "match_rate",
        "window_to_match_conversion_rate",
        "unmatched_launches_inside_window",
        "outside_weather_window_launches",
        "median_abs_diff_minutes",
    ]
]

,facility,country_code,launches,station_ids,nearest_downloaded_station_id,nearest_downloaded_station_distance_miles,launches_in_weather_window,date_window_coverage_rate,matched_launches,match_rate,window_to_match_conversion_rate,unmatched_launches_inside_window,outside_weather_window_launches,median_abs_diff_minutes
0,Plesetsk Cosmodrome,RU,1648,RSM00022657,RSM00022657,11.653629,1240,0.752427,1091.0,0.662015,0.879839,149.0,408,45.0
1,Baikonur Cosmodrome,KZ,1525,KZM00035953,KZM00035953,49.995056,843,0.552787,145.0,0.095082,0.172005,698.0,682,50.0
2,Cape Canaveral Space Force Station,US,810,"USL000TRDF1, USW00012868",USW00012868,1.327739,810,1.000000,509.0,0.628395,0.628395,301.0,0,14.0
3,Vandenberg Space Force Base,US,711,"USW00093214, USW00093215",USW00093215,1.193786,711,1.000000,672.0,0.945148,0.945148,39.0,0,13.0
4,Guiana SC,GF,311,FGI0000SOCA,FGI0000SOCA,36.558001,306,0.983923,304.0,0.977492,0.993464,2.0,5,11.0
5,Kennedy Space Center,US,192,"USW00012886, USW00092821",USW00012886,3.868927,175,0.911458,173.0,0.901042,0.988571,2.0,17,0.0
6,Xichang Satellite LC,CN,167,CHM00056571,CHM00056571,1.060992,167,1.000000,167.0,1.000000,1.000000,0.0,0,53.0
7,Jiuquan Satellite LC,CN,158,CHA00524460,CHA00524460,45.711034,29,0.183544,25.0,0.158228,0.862069,4.0,129,40.0
8,Taiyuan Satellite LC,CN,99,CHM00053663,CHM00053663,12.993011,11,0.111111,0.0,0.000000,0.000000,11.0,88,NaN
9,Kapustin Yar,RU,97,RSM00034571,RSM00034571,1.861946,75,0.773196,54.0,0.556701,0.720000,21.0,22,58.0


## Facility Coverage Overview

In [174]:
facility_coverage_overview = (
    facility_summary.merge(
        facility_date_coverage[
            [
                "facility",
                "file_exists",
                "file_name",
                "station_ids",
                "utc_offset_hours",
                "weather_start_lstd",
                "weather_end_lstd",
                "launches_in_weather_window",
                "launches_before_weather_start",
                "launches_after_weather_end",
                "date_window_coverage_rate",
            ]
        ],
        on="facility",
        how="left",
    )
    .merge(
        weather_match_coverage[
            [
                "facility",
                "matched_launches",
                "match_rate",
                "median_abs_diff_minutes",
                "weather_rows_after_cleaning",
            ]
        ],
        on="facility",
        how="left",
    )
    .merge(
        downloaded_station_distance_summary[
            [
                "facility",
                "downloaded_station_count",
                "nearest_downloaded_station_id",
                "nearest_downloaded_station_name",
                "nearest_downloaded_station_distance_miles",
                "farthest_downloaded_station_distance_miles",
                "best_distance_rank_among_all_lcdv2_stations",
            ]
        ],
        on="facility",
        how="left",
    )
    .sort_values(["launches", "file_exists", "matched_launches"], ascending=[False, False, False])
    .reset_index(drop=True)
)

facility_coverage_overview

,facility,country,country_code,launches,first_launch_utc,last_launch_utc,first_launch_date,last_launch_date,raw_location_strings,launch_site_labels,facility_lat,facility_lon,coordinate_note,file_exists,file_name,station_ids,utc_offset_hours,weather_start_lstd,weather_end_lstd,launches_in_weather_window,launches_before_weather_start,launches_after_weather_end,date_window_coverage_rate,matched_launches,match_rate,median_abs_diff_minutes,weather_rows_after_cleaning,downloaded_station_count,nearest_downloaded_station_id,nearest_downloaded_station_name,nearest_downloaded_station_distance_miles,farthest_downloaded_station_distance_miles,best_distance_rank_among_all_lcdv2_stations
0,Plesetsk Cosmodrome,Russia,RU,1648,1966-03-17 10:28:00+00:00,2021-12-27 19:00:00+00:00,1966-03-17,2021-12-27,11,1,62.926415,40.555239,Locations.csv,True,Plesetsk_Cosmodrome.csv,RSM00022657,3.0,1966-01-01 00:00:00,1988-10-08 15:00:00,1240,0,408,0.752427,1091.0,0.662015,45.0,57732.0,1.0,RSM00022657,EMCA,11.653629,11.653629,1.0
1,Baikonur Cosmodrome,Kazakhstan,KZ,1525,1957-10-04 19:28:00+00:00,2021-12-27 13:10:00+00:00,1957-10-04,2021-12-27,21,1,45.964585,63.305243,Locations.csv,True,Baikonur_Cosmodrome.csv,KZM00035953,5.0,1966-01-01 02:00:00,1991-12-21 02:00:00,843,128,554,0.552787,145.0,0.095082,50.0,13176.0,1.0,KZM00035953,DZUSALY,49.995056,49.995056,1.0
2,Cape Canaveral Space Force Station,United States,US,810,1957-12-06 16:44:00+00:00,2021-12-19 03:58:00+00:00,1957-12-06,2021-12-19,22,1,28.502515,-80.566468,Locations.csv,True,cape_canaveral_sfs.csv,"USL000TRDF1, USW00012868",-5.0,1957-12-01 00:00:00,2021-12-31 23:56:00,810,0,0,1.000000,509.0,0.628395,14.0,459992.0,2.0,USW00012868,CAPE KENNEDY AFS 74794,1.327739,5.760230,1.0
3,Vandenberg Space Force Base,United States,US,711,1959-02-28 21:49:00+00:00,2021-12-18 12:41:00+00:00,1959-02-28,2021-12-18,16,1,34.683069,-120.590025,Locations.csv,True,vandenberg_sfb.csv,"USW00093214, USW00093215",-8.0,1959-01-01 00:00:00,2021-12-31 23:58:00,711,0,0,1.000000,672.0,0.945148,13.0,624228.0,2.0,USW00093215,POINT ARGUELLO AFWB,1.193786,2.674893,1.0
4,Guiana SC,French Guiana,GF,311,1970-03-10 12:20:00+00:00,2021-12-25 12:20:00+00:00,1970-03-10,2021-12-25,6,1,5.201590,-52.728131,Locations.csv,True,Guiana_SC.csv,FGI0000SOCA,-3.0,1972-12-31 23:00:00,2021-12-31 23:30:00,306,5,0,0.983923,304.0,0.977492,11.0,483251.0,1.0,FGI0000SOCA,ROCHAMBEAU,36.558001,36.558001,1.0
5,Kennedy Space Center,United States,US,192,1967-11-09 12:00:00+00:00,2021-12-21 10:07:00+00:00,1967-11-09,2021-12-21,2,1,28.626383,-80.620470,Locations.csv,True,kennedy_sc.csv,"USW00012886, USW00092821",-5.0,1978-03-16 19:00:00,2021-12-31 23:59:00,175,17,0,0.911458,173.0,0.901042,0.0,1978955.0,2.0,USW00012886,TITUSVILLE NASA SHUTTLE FAC,3.868927,4.447363,1.0
6,Xichang Satellite LC,China,CN,167,1984-01-29 12:25:00+00:00,2021-12-29 16:43:00+00:00,1984-01-29,2021-12-29,3,1,27.893551,102.250931,Locations.csv,True,Xichang_Satellite_LC.csv,CHM00056571,8.0,1984-01-01 00:00:00,2021-12-31 23:59:00,167,0,0,1.000000,167.0,1.000000,53.0,110869.0,1.0,CHM00056571,XICHANG,1.060992,1.060992,1.0
7,Jiuquan Satellite LC,China,CN,158,1970-04-24 13:35:00+00:00,2021-12-29 11:13:00+00:00,1970-04-24,2021-12-29,6,1,40.984524,100.208695,Locations.csv,True,Jiuquan_Satellite_LC.csv,CHA00524460,8.0,1973-01-01 08:00:00,2002-05-13 17:00:00,29,2,127,0.183544,25.0,0.158228,40.0,54725.0,1.0,CHA00524460,SHUANGCHENGTZU,45.711034,45.711034,1.0
8,Taiyuan Satellite LC,China,CN,99,1988-09-06 20:30:00+00:00,2021-12-26 03:11:00+00:00,1988-09-06,2021-12-26,4,1,38.848577,111.607983,Locations.csv,True,Taiyuan_Satellite_LC.csv,CHM00053663,8.0,2001-11-18 08:00:00,2007-07-27 20:00:00,11,12,76,0.111111,0.0,0.000000,NaN,5.0,1.0,CHM00053663,WU-ZHAI,12.993011,12.993011,2.0
9,Kapustin Yar,Russia,RU,97,1961-10-27 16:30:00+00:00,1999-04-28 20:30:00+00:00,1961-10-27,1999-04-28,3,1,48.575267,45.766175,Locations.csv,True,Kapustin_Yar.csv,RSM00034571,4.0,1961-01-01 03:00:00,1974-03-10 00:00:00,75,0,22,0.773196,54.0,0.55

## Count Of Launches With Weather Data Coverage

In [175]:
launch_weather_coverage_counts = pd.DataFrame(
    [
        {
            "coverage_definition": "total launches",
            "launch_count": int(facility_coverage_overview["launches"].sum()),
        },
        {
            "coverage_definition": "launches at facilities with a downloaded NOAA file",
            "launch_count": int(
                facility_coverage_overview.loc[
                    facility_coverage_overview["file_exists"].fillna(False),
                    "launches",
                ].sum()
            ),
        },
        {
            "coverage_definition": "launches inside downloaded weather date windows",
            "launch_count": int(
                facility_coverage_overview["launches_in_weather_window"].fillna(0).sum()
            ),
        },
        {
            "coverage_definition": "launches matched to weather within 2 hours",
            "launch_count": int(
                facility_coverage_overview["matched_launches"].fillna(0).sum()
            ),
        },
    ]
)

launch_weather_coverage_counts["share_of_total_launches"] = (
    launch_weather_coverage_counts["launch_count"]
    / launch_weather_coverage_counts.loc[
        launch_weather_coverage_counts["coverage_definition"] == "total launches",
        "launch_count",
    ].iloc[0]
)

launch_weather_coverage_counts

,coverage_definition,launch_count,share_of_total_launches
0,total launches,6168,1.000000
1,launches at facilities with a downloaded NOAA file,6150,0.997082
2,launches inside downloaded weather date windows,4690,0.760376
3,launches matched to weather within 2 hours,3367,0.545882


## Remaining Gaps

In [176]:
missing_or_partial = facility_coverage_overview[
    [
        "facility",
        "country_code",
        "launches",
        "file_exists",
        "station_ids",
        "first_launch_date",
        "last_launch_date",
        "weather_start_lstd",
        "weather_end_lstd",
        "launches_in_weather_window",
        "matched_launches",
        "match_rate",
    ]
].copy()

missing_or_partial["fully_covered_by_date_window"] = (
    missing_or_partial["launches_in_weather_window"] == missing_or_partial["launches"]
)
missing_or_partial["fully_matched_within_2h"] = (
    missing_or_partial["matched_launches"] == missing_or_partial["launches"]
)

missing_or_partial.loc[
    (~missing_or_partial["file_exists"])
    | (~missing_or_partial["fully_covered_by_date_window"])
    | (~missing_or_partial["fully_matched_within_2h"])
].sort_values(["launches", "file_exists", "matched_launches"], ascending=[False, True, True])

,facility,country_code,launches,file_exists,station_ids,first_launch_date,last_launch_date,weather_start_lstd,weather_end_lstd,launches_in_weather_window,matched_launches,match_rate,fully_covered_by_date_window,fully_matched_within_2h
0,Plesetsk Cosmodrome,RU,1648,True,RSM00022657,1966-03-17,2021-12-27,1966-01-01 00:00:00,1988-10-08 15:00:00,1240,1091.0,0.662015,False,False
1,Baikonur Cosmodrome,KZ,1525,True,KZM00035953,1957-10-04,2021-12-27,1966-01-01 02:00:00,1991-12-21 02:00:00,843,145.0,0.095082,False,False
2,Cape Canaveral Space Force Station,US,810,True,"USL000TRDF1, USW00012868",1957-12-06,2021-12-19,1957-12-01 00:00:00,2021-12-31 23:56:00,810,509.0,0.628395,True,False
3,Vandenberg Space Force Base,US,711,True,"USW00093214, USW00093215",1959-02-28,2021-12-18,1959-01-01 00:00:00,2021-12-31 23:58:00,711,672.0,0.945148,True,False
4,Guiana SC,GF,311,True,FGI0000SOCA,1970-03-10,2021-12-25,1972-12-31 23:00:00,2021-12-31 23:30:00,306,304.0,0.977492,False,False
5,Kennedy Space Center,US,192,True,"USW00012886, USW00092821",1967-11-09,2021-12-21,1978-03-16 19:00:00,2021-12-31 23:59:00,175,173.0,0.901042,False,False
7,Jiuquan Satellite LC,CN,158,True,CHA00524460,1970-04-24,2021-12-29,1973-01-01 08:00:00,2002-05-13 17:00:00,29,25.0,0.158228,False,False
8,Taiyuan Satellite LC,CN,99,True,CHM00053663,1988-09-06,2021-12-26,2001-11-18 08:00:00,2007-07-27 20:00:00,11,0.0,0.000000,False,False
9,Kapustin Yar,RU,97,True,RSM00034571,1961-10-27,1999-04-28,1961-01-01 03:00:00,1974-03-10 00:00:00,75,54.0,0.556701,False,False
10,Tanegashima SC,JP,85,True,JAI0000RJFG,1975-09-09,2021-12-22,1975-01-01 08:00:00,2021-12-31 18:00:00,85,67.0,0.788235,True,False


## Missing Facilities With LCDv2 Stations Within 50 Miles

In [177]:
missing_facilities = facility_coverage_overview.loc[
    ~facility_coverage_overview["file_exists"].fillna(False)
].copy()

missing_station_radius_rows = []
for _, row in missing_facilities.iterrows():
    distances = haversine_miles(
        row["facility_lat"],
        row["facility_lon"],
        stations["station_lat"].to_numpy(),
        stations["station_lon"].to_numpy(),
    )
    nearby = stations.copy()
    nearby["distance_miles"] = distances
    nearby = nearby.loc[nearby["distance_miles"] <= 50].sort_values("distance_miles").copy()

    if nearby.empty:
        missing_station_radius_rows.append(
            {
                "facility": row["facility"],
                "country_code": row["country_code"],
                "launches": row["launches"],
                "first_launch_date": row["first_launch_date"],
                "last_launch_date": row["last_launch_date"],
                "facility_lat": row["facility_lat"],
                "facility_lon": row["facility_lon"],
                "stations_within_50_miles": 0,
                "nearest_station_id": pd.NA,
                "nearest_station_name": pd.NA,
                "nearest_station_lat": pd.NA,
                "nearest_station_lon": pd.NA,
                "nearest_distance_miles": pd.NA,
                "station_ids_within_50_miles": "",
            }
        )
        continue

    nearest = nearby.iloc[0]
    missing_station_radius_rows.append(
        {
            "facility": row["facility"],
            "country_code": row["country_code"],
            "launches": row["launches"],
            "first_launch_date": row["first_launch_date"],
            "last_launch_date": row["last_launch_date"],
            "facility_lat": row["facility_lat"],
            "facility_lon": row["facility_lon"],
            "stations_within_50_miles": int(len(nearby)),
            "nearest_station_id": nearest["station_id"],
            "nearest_station_name": nearest["station_name"],
            "nearest_station_lat": nearest["station_lat"],
            "nearest_station_lon": nearest["station_lon"],
            "nearest_distance_miles": nearest["distance_miles"],
            "station_ids_within_50_miles": " | ".join(nearby["station_id"].tolist()),
        }
    )

missing_facilities_within_50_miles = pd.DataFrame(missing_station_radius_rows).sort_values(
    ["launches", "stations_within_50_miles", "nearest_distance_miles"],
    ascending=[False, False, True],
).reset_index(drop=True)

missing_facilities_within_50_miles

,facility,country_code,launches,first_launch_date,last_launch_date,facility_lat,facility_lon,stations_within_50_miles,nearest_station_id,nearest_station_name,nearest_station_lat,nearest_station_lon,nearest_distance_miles,station_ids_within_50_miles
0,San Marco LP,KE,9,1967-04-26,1988-03-25,-2.995738,40.194845,0,<NA>,<NA>,<NA>,<NA>,<NA>,
1,Hammaguir,DZ,4,1965-11-26,1967-02-15,30.883284,-3.037591,0,<NA>,<NA>,<NA>,<NA>,<NA>,
2,Barents Sea LA,RU,3,1998-07-07,2006-05-26,74.988392,37.106368,0,<NA>,<NA>,<NA>,<NA>,<NA>,
3,DeBo 3 Barge,NaN,1,2020-09-15,2020-09-15,35.494277,123.796461,0,<NA>,<NA>,<NA>,<NA>,<NA>,
4,Tai Rui Barge,NaN,1,2019-06-05,2019-06-05,35.494277,123.796461,0,<NA>,<NA>,<NA>,<NA>,<NA>,


## Missing Facilities That Do Have At Least One Station Within 50 Miles

In [178]:
missing_facilities_within_50_miles.loc[
    missing_facilities_within_50_miles["stations_within_50_miles"] > 0
].reset_index(drop=True)

,facility,country_code,launches,first_launch_date,last_launch_date,facility_lat,facility_lon,stations_within_50_miles,nearest_station_id,nearest_station_name,nearest_station_lat,nearest_station_lon,nearest_distance_miles,station_ids_within_50_miles


## Export Updated Coverage Tables

In [179]:
noaa_file_inventory.to_csv(OUTPUT_DIR / "noaa_file_inventory.csv", index=False)
downloaded_station_summary.to_csv(OUTPUT_DIR / "downloaded_station_summary.csv", index=False)
downloaded_station_distance_summary.to_csv(OUTPUT_DIR / "downloaded_station_distance_summary.csv", index=False)
three_closest_stations.to_csv(OUTPUT_DIR / "lcdv2_three_closest_stations_by_launch_facility.csv", index=False)
facility_date_coverage.to_csv(OUTPUT_DIR / "noaa_facility_date_window_coverage.csv", index=False)
weather_match_coverage.to_csv(OUTPUT_DIR / "noaa_weather_match_coverage.csv", index=False)
join_quality_summary.to_csv(OUTPUT_DIR / "noaa_join_quality_summary.csv", index=False)
facility_coverage_overview.to_csv(OUTPUT_DIR / "noaa_facility_coverage_overview.csv", index=False)
launch_weather_coverage_counts.to_csv(OUTPUT_DIR / "launch_weather_coverage_counts.csv", index=False)
missing_facilities_within_50_miles.to_csv(
    OUTPUT_DIR / "missing_facilities_with_lcdv2_stations_within_50_miles.csv",
    index=False,
)

pd.DataFrame(
    {
        "file": [
            str(OUTPUT_DIR / "noaa_file_inventory.csv"),
            str(OUTPUT_DIR / "downloaded_station_summary.csv"),
            str(OUTPUT_DIR / "downloaded_station_distance_summary.csv"),
            str(OUTPUT_DIR / "lcdv2_three_closest_stations_by_launch_facility.csv"),
            str(OUTPUT_DIR / "noaa_facility_date_window_coverage.csv"),
            str(OUTPUT_DIR / "noaa_weather_match_coverage.csv"),
            str(OUTPUT_DIR / "noaa_join_quality_summary.csv"),
            str(OUTPUT_DIR / "noaa_facility_coverage_overview.csv"),
            str(OUTPUT_DIR / "launch_weather_coverage_counts.csv"),
            str(OUTPUT_DIR / "missing_facilities_with_lcdv2_stations_within_50_miles.csv"),
        ]
    }
)

,file
0,data_final\derived\noaa_file_inventory.csv
1,data_final\derived\downloaded_station_summary.csv
2,data_final\derived\downloaded_station_distance_summary.csv
3,data_final\derived\lcdv2_three_closest_stations_by_launch_facility.csv
4,data_final\derived\noaa_facility_date_window_coverage.csv
5,data_final\derived\noaa_weather_match_coverage.csv
6,data_final\derived\noaa_join_quality_summary.csv
7,data_final\derived\noaa_facility_coverage_overview.csv
8,data_final\derived\launch_weather_coverage_counts.csv
9,data_final\derived\missing_facilities_with_lcdv2_stations_within_50_miles.csv


## Quick Totals

In [180]:
summary = pd.Series(
    {
        "facilities_in_launch_data": int(len(facility_summary)),
        "facilities_with_downloaded_noaa_file": int(facility_coverage_overview["file_exists"].fillna(False).sum()),
        "launches_total": int(facility_coverage_overview["launches"].sum()),
        "launches_with_downloaded_file": int(
            facility_coverage_overview.loc[facility_coverage_overview["file_exists"].fillna(False), "launches"].sum()
        ),
        "launches_inside_downloaded_weather_window": int(
            facility_coverage_overview["launches_in_weather_window"].fillna(0).sum()
        ),
        "launches_matched_within_2h": int(
            facility_coverage_overview["matched_launches"].fillna(0).sum()
        ),
    }
)

summary

facilities_in_launch_data                      40
facilities_with_downloaded_noaa_file           35
launches_total                               6168
launches_with_downloaded_file                6150
launches_inside_downloaded_weather_window    4690
launches_matched_within_2h                   3367
dtype: int64